In [1]:
import pandas as pd
import re
import numpy as np

In [39]:
ucla_ground_truth = pd.read_excel('../inputs/ucla_ground_truth.xlsx', index_col=0)  
und_ground_truth = pd.read_excel('../inputs/und_ground_truth.xlsx', index_col=0)  
uvm_ground_truth = pd.read_excel('../inputs/uvm_ground_truth.xlsx', index_col=0)

df_ground_truth = pd.concat([ucla_ground_truth.assign(ipeds_id='110662'), und_ground_truth.assign(ipeds_id='200280'), uvm_ground_truth.assign(ipeds_id='231174')], ignore_index=True)
df_ground_truth.head(5)


,cat_type,start_yr,end_yr,page_num,col_num,type,annote_id,Department,Number,Title,...,Prerequisites,Credits,Teacher,decade,quinquennium,DepartmentCleaned,combined_text,has_ai,is_not_course,ipeds_id
0,both,1883,1884,26,0,courses,784005,Science,28,"Physiology, from skeleton and models, with law...",...,NaN,0,"{'Honorific': '', 'Name': 'Normal School', 'De...",1880,1880,science,"Physiology, from skeleton and models, with law...",0,0,110662
1,both,1883,1884,26,0,courses,784005,Language,28,Spelling,...,NaN,0,"{'Honorific': '', 'Name': 'Normal School', 'De...",1880,1880,language,Spelling,0,0,110662
2,both,1883,1884,26,0,courses,784005,Middle Year,28,FIRST TERM.,...,NaN,0,"{'Honorific': '', 'Name': 'Normal School', 'De...",1880,1880,middle year,FIRST TERM.,0,1,110662
3,both,1883,1884,26,0,courses,784005,Language,28,Rules of construction,...,NaN,0,"{'Honorific': '', 'Name': 'Normal School', 'De...",1880,1880,language,Rules of construction Letter writing.,0,0,110662
4,both,1883,1884,26,0,courses,784005,Science,28,"Physical Geography-A review of contour, and pr...",...,NaN,0,"{'Honorific': '', 'Name': 'Normal School', 'De...",1880,1880,science,"Physical Geography-A review of contour, and pr...",0,0,110662


In [40]:
## Preprocessing functions to apply to catalogs' aggregated course descriptions
def remove_double_spaces(text):
    if not text:
        return ""
    return " ".join(text.split())


def remove_paragraphs_dash(text):
    return re.sub(r"-\n", "", text)


def remove_paragraphs(text):
    return re.sub("\n", " ", text)


In [41]:
def clean_text(series_text):
    series_text = series_text.apply(remove_paragraphs_dash)
    series_text = series_text.apply(remove_paragraphs)
    series_text = series_text.apply(remove_double_spaces)
    series_text = series_text.str.strip().replace(
        ["nan", "None", ""], np.nan
    )
    series_text = series_text.fillna('').apply(lambda text: text if len(str(text)) >= 5 else np.nan)
    return series_text.str.title()

In [75]:
cond_ipeds_id = df_ground_truth["ipeds_id"] == "110662"
cond_cat_type = df_ground_truth["cat_type"] == "both"
cond_start_year = df_ground_truth["start_yr"] == 1999

df_ground_truth.loc[cond_ipeds_id & cond_cat_type & cond_start_year, "end_yr"] = 2001

In [76]:
###in the DB there is already a id_catalog. for this NLP project purpose we created it here
df_ground_truth["id_catalog"] = (
    df_ground_truth["ipeds_id"].astype(str)
    + "_"
    + df_ground_truth["cat_type"].astype(str)
    + "_"
    + df_ground_truth["start_yr"].astype(str)
    + "_"
    + df_ground_truth["end_yr"].astype(str)
)

In [77]:
df_ground_truth["Description"] = clean_text(df_ground_truth["Description"].fillna(''))

In [78]:
df_ground_truth["Title"] = clean_text(df_ground_truth["Title"].fillna(''))

In [79]:
df_top200_departments = pd.read_csv('../../DataPreprocessing/inputs/top200_department_mapping.csv',sep=';')
map_departmentcleaned_corrected = df_top200_departments.set_index('DepartmentCleaned')['Department_corrected'].to_dict()
df_ground_truth['Department'] = df_ground_truth['DepartmentCleaned'].replace('0',None).fillna('').map(map_departmentcleaned_corrected).fillna(df_ground_truth['DepartmentCleaned'])

In [80]:
df_ground_truth["id_course"] = (
    df_ground_truth["Number"].astype(str).fillna('') + "_" + df_ground_truth["Title"].fillna('')
)

In [81]:
df_ground_truth["id_dep_code"] = (
    df_ground_truth["Department"].fillna('') + "_" + df_ground_truth["Number"].fillna('')
)

In [82]:
df_ground_truth["id_department_course"] = (
    df_ground_truth["Department"].fillna('') + "_" + df_ground_truth["Title"].fillna('')
)

In [83]:
df_ground_truth["Description_len"] = df_ground_truth["Description"].apply(lambda x: len(x) if pd.notnull(x) else 0)

In [84]:
df_ground_truth_unique = df_ground_truth.sort_values(by='Description_len', ascending=False).drop_duplicates(["id_department_course", "id_catalog"],keep='first')

In [85]:
df_final = df_ground_truth_unique.drop(['Description_len'],axis=1).copy()

In [86]:
df_final.to_pickle('../outputs/ground_truth_processed.pkl')

In [87]:
df_final.shape

(2971, 25)